# Module 1 — Model Evaluation on Zenodo Test Set

Evaluates `best_model.pt` (epoch 8, mIoU=0.8135) in two inference modes:

| Mode | Band 3 | Matches training? |
|---|---|---|
| **A — Alpha** | `arctan2(√p₂,√p₁)×2×(180/π)/90` | ✅ Yes |
| **B — RVI_dp** | `4·VH/(VV+VH)/2` | ❌ No — code-only change, never trained |

## ⚠️ BEFORE YOU RUN
1. **Enable GPU**: Settings → Accelerator → GPU T4 x1 → Save  
2. **Add Secret**: Account → Settings → Secrets → `HF_TOKEN`  
3. **Attach 4 datasets** via + Add Data → search `rohithsheregar/sentinel1-sar-*`:
   - `sentinel1-sar-oil-spill-test` (900 TIFFs)
   - `sentinel1-sar-oil-spill-train` (2400 TIFFs)
   - `sentinel1-sar-lookalike-train` (1370 TIFFs)
   - `sentinel1-sar-no-oil-train` (1370 TIFFs)

In [ ]:
# Cell 0 — GPU Verification
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Fix: Settings -> Accelerator -> GPU T4 x1 -> Save -> Factory Reset session."
    )

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}  "
          f"({torch.cuda.get_device_properties(i).total_memory/1024**3:.1f} GB)")
print(f"CUDA: {torch.version.cuda}   PyTorch: {torch.__version__}")

In [ ]:
# Cell 1 — Setup: Repo / Dependencies / HF Connection
import os, sys, time

HF_REPO_ID  = "RohithSheregar/oil-spill-models"
HF_FILENAME = "best_model.pt"

# 1a. Clone / pull latest repo (same URL as module-1-training.ipynb)
REPO_URL = "https://github.com/Rohith-Sheregar/Oil-Spill-Detection-New.git"
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Repo: {os.getcwd()}")

# 1b. Install dependencies
print("Installing deps...")
os.system("pip install -q segmentation-models-pytorch scikit-image scipy joblib imagecodecs huggingface_hub")
print("Dependencies ready.")

# 1c. Load HF Token from Kaggle Secret
HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    print(f"Could not load HF_TOKEN: {e}")
    print("Add it: Account -> Settings -> Secrets -> HF_TOKEN")

# 1d. HF connection test
if HF_TOKEN:
    from huggingface_hub import HfApi
    api  = HfApi(token=HF_TOKEN)
    user = api.whoami()["name"]
    print(f"Authenticated as: {user}")
    api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=True)
    print(f"Repo ready: {HF_REPO_ID}")
    # Quick test upload
    _tf = f"/kaggle/working/_hf_test_{int(time.time())}.txt"
    open(_tf, "w").write("ok")
    api.upload_file(path_or_fileobj=_tf, path_in_repo=os.path.basename(_tf),
                    repo_id=HF_REPO_ID, commit_message="Eval connection test")
    api.delete_file(path_in_repo=os.path.basename(_tf), repo_id=HF_REPO_ID)
    os.remove(_tf)
    print("HF Hub connected — OK")
else:
    print("WARNING: No HF_TOKEN — model download may fail for private repos.")

In [ ]:
# Cell 2 — Data Symlinks (same logic as module-1-training.ipynb Cell 2)
# Verified paths from training notebook output (2026-07-30):
#   /kaggle/input/datasets/rohithsheregar/
#     sentinel1-sar-oil-spill-test    (900 TIFFs)
#     sentinel1-sar-oil-spill-train   (2400 TIFFs)
#     sentinel1-sar-lookalike-train   (1370 TIFFs)
#     sentinel1-sar-no-oil-train      (1370 TIFFs)
import glob, shutil
from pathlib import Path

INPUT_DIR = "/kaggle/input/datasets/rohithsheregar"
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = "/kaggle/input/datasets" if os.path.exists("/kaggle/input/datasets") else "/kaggle/input"

print(f"Input root: {INPUT_DIR}")
available = os.listdir(INPUT_DIR)
for d in available:
    n = len(glob.glob(os.path.join(INPUT_DIR, d, "**", "*.tif*"), recursive=True))
    print(f"  {d}  ({n} TIFFs)")

WORK_DATA = "/kaggle/working/data"
if os.path.exists(WORK_DATA):
    shutil.rmtree(WORK_DATA)

mappings = {
    "test":            lambda n: "test" in n,
    "train/oil":       lambda n: "oil" in n and not any(k in n for k in ["lookalike","no","test"]),
    "train/lookalike": lambda n: "lookalike" in n,
    "train/no_oil":    lambda n: "no" in n and "oil" in n,
}

print("\nCreating symlinks...")
for subpath, cond in mappings.items():
    matched = [d for d in available if cond(d.lower())]
    if not matched:
        print(f"  WARNING: no match for '{subpath}'")
        continue
    src = os.path.join(INPUT_DIR, matched[0])
    dst = os.path.join(WORK_DATA, subpath)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    os.symlink(src, dst)
    n = len(glob.glob(os.path.join(src, "**", "*.tif*"), recursive=True))
    print(f"  {subpath} -> {matched[0]}  ({n} TIFFs)")

TEST_DIR = Path(WORK_DATA) / "test"
n_test   = len(list(TEST_DIR.rglob("*.tif*")))
print(f"\nTest dir: {TEST_DIR}  ({n_test} TIFFs)")
assert n_test > 0, "No TIFFs found — attach all 4 datasets before running!"

from src.training.zenodo_sos_dataset import discover_sos_pairs
test_df = discover_sos_pairs(TEST_DIR)
print(f"Test pairs found: {len(test_df)}")
print(test_df["class_name"].value_counts().to_string())
print(f"Sample: {test_df.iloc[0].scene_id} | {test_df.iloc[0].image_path}")

In [ ]:
# Cell 3 — Pull best_model.pt from HuggingFace & Load Model
from huggingface_hub import HfFileSystem, hf_hub_download
import torch
from src.models.deeplab_scse import DeepLabV3PlusSCSE

CKPT_DIR = "/kaggle/working/eval_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
ckpt_path = os.path.join(CKPT_DIR, HF_FILENAME)

if os.path.exists(ckpt_path):
    print(f"Already present: {ckpt_path}  ({os.path.getsize(ckpt_path)/1e6:.1f} MB)")
else:
    if HF_TOKEN:
        _fs = HfFileSystem(token=HF_TOKEN)
        _pts = [f.split("/")[-1] for f in _fs.ls(HF_REPO_ID, detail=False) if f.endswith(".pt")]
        print(f"Available checkpoints: {_pts}")

    print(f"Downloading {HF_FILENAME}...")
    ckpt_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=HF_FILENAME,
        token=HF_TOKEN or None,
        local_dir=CKPT_DIR,
    )
    print(f"Downloaded: {ckpt_path}  ({os.path.getsize(ckpt_path)/1e6:.1f} MB)")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt   = torch.load(ckpt_path, map_location=device, weights_only=False)

print("\nCheckpoint info:")
for k, v in ckpt.items():
    if k == "model_state":
        print(f"  {k}: <{len(v)} tensors>")
    elif k == "config":
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: {v}")

model = DeepLabV3PlusSCSE(in_channels=5, classes=1, input_size=256)
model.load_state_dict(ckpt["model_state"])
model.to(device).eval()

print(f"\nModel loaded — {sum(p.numel() for p in model.parameters()):,} params")
print(f"  Epoch  : {ckpt.get('epoch','?')}")
print(f"  mIoU   : {ckpt.get('val_miou','?'):.4f}")
print(f"  Loss   : {ckpt.get('val_loss','?'):.4f}")

In [ ]:
# Cell 4 — Inference Helpers
import numpy as np
import tifffile
from src.training.zenodo_sos_dataset import read_mask, robust_normalize
from src.preprocessing.polsar_decomp import db_to_linear, dual_pol_entropy_alpha, compute_rvi_dp
from src.preprocessing.wind_ratio import compute_wind_corrected_ratio


def _as_hwc(arr):
    arr = np.asarray(arr)
    if arr.ndim == 2:
        return arr[..., None]
    if arr.shape[0] <= 8 and arr.shape[1] > 32 and arr.shape[2] > 32:
        return np.moveaxis(arr, 0, -1)
    return arr


def _pstretch(arr):
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        return np.zeros_like(arr, dtype=np.float32)
    lo, hi = np.percentile(finite, [1.0, 99.0])
    if hi <= lo:
        return np.clip(arr - lo, 0.0, None).astype(np.float32)
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)


def _vv_vh(path):
    hwc = _as_hwc(tifffile.imread(path)).astype(np.float32)
    if hwc.shape[-1] < 2:
        hwc = np.repeat(hwc, 2, axis=-1)
    return hwc[..., 0], hwc[..., 1]


def build_alpha(path, ws=7.0, inc=35.0, wd=0.0):
    "Mode A: Band3 = alpha/90  (matches training)"
    vv, vh       = _vv_vh(path)
    vvl, vhl     = db_to_linear(vv), db_to_linear(vh)
    H, alpha     = dual_pol_entropy_alpha(vvl, vhl)
    band_a       = (alpha / 90.0).astype(np.float32)
    w            = compute_wind_corrected_ratio(vv, vh, wind_speed_ms=ws, incidence_deg=inc, wind_dir_deg=wd)
    return np.stack([_pstretch(vv), _pstretch(vh), H, band_a, w], axis=0)


def build_rvi(path, ws=7.0, inc=35.0, wd=0.0):
    "Mode B: Band3 = RVI_dp/2  (mismatch — never trained on)"
    vv, vh       = _vv_vh(path)
    vvl, vhl     = db_to_linear(vv), db_to_linear(vh)
    H, _         = dual_pol_entropy_alpha(vvl, vhl)
    rvi          = np.clip(compute_rvi_dp(vvl, vhl) / 2.0, 0.0, 1.0).astype(np.float32)
    w            = compute_wind_corrected_ratio(vv, vh, wind_speed_ms=ws, incidence_deg=inc, wind_dir_deg=wd)
    return np.stack([_pstretch(vv), _pstretch(vh), H, rvi, w], axis=0)


@torch.no_grad()
def infer_scene(model, chw, ps=256, thr=0.5):
    C, H, W = chw.shape
    acc = np.zeros((H, W), np.float32)
    cnt = np.zeros((H, W), np.float32)
    stride = ps // 2
    ys = list(range(0, max(H-ps, 0)+1, stride))
    xs = list(range(0, max(W-ps, 0)+1, stride))
    if not ys or ys[-1]+ps < H: ys.append(max(0, H-ps))
    if not xs or xs[-1]+ps < W: xs.append(max(0, W-ps))
    ph, pw = max(0, ps-H), max(0, ps-W)
    if ph or pw:
        chw = np.pad(chw, ((0,0),(0,ph),(0,pw)), mode="edge")
        acc = np.pad(acc, ((0,ph),(0,pw)), mode="constant")
        cnt = np.pad(cnt, ((0,ph),(0,pw)), mode="constant")
    norm = robust_normalize(chw)
    for y in ys:
        for x in xs:
            t = torch.from_numpy(norm[:, y:y+ps, x:x+ps][None].astype(np.float32)).to(device)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                p = torch.sigmoid(model(t)).squeeze().cpu().numpy()
            acc[y:y+ps, x:x+ps] += p
            cnt[y:y+ps, x:x+ps] += 1.0
    prob = (acc / np.maximum(cnt, 1e-6))[:H, :W]
    return (prob >= thr).astype(np.uint8), prob


def get_metrics(pred, gt):
    tp = int(np.logical_and(pred==1, gt==1).sum())
    tn = int(np.logical_and(pred==0, gt==0).sum())
    fp = int(np.logical_and(pred==1, gt==0).sum())
    fn = int(np.logical_and(pred==0, gt==1).sum())
    pr = tp / max(tp+fp, 1)
    rc = tp / max(tp+fn, 1)
    f1 = 2*pr*rc / max(pr+rc, 1e-6)
    miou = ((tp/max(tp+fp+fn,1)) + (tn/max(tn+fp+fn,1))) / 2.0
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, precision=pr, recall=rc, f1=f1, miou=miou)


print("Helpers ready.")

In [ ]:
# Cell 5 — Run Evaluation (both modes on all test scenes)
# Estimated time: ~2-4 hours on T4 GPU for 900 scenes x 2 modes
# Run as Save & Run All (9-hour session limit)
import pandas as pd
from tqdm.auto import tqdm

OUTPUT_DIR = Path("/kaggle/working/eval_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rows_a, rows_b, viz = [], [], []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating"):
    try:
        hwc      = _as_hwc(tifffile.imread(row.image_path)).astype(np.float32)
        H_px, W_px = hwc.shape[:2]
        gt       = read_mask(row.mask_path if row.mask_path else None, (H_px, W_px)).astype(np.uint8)

        # Mode A: alpha/90 — correct, matches training
        pa, proba = infer_scene(model, build_alpha(row.image_path))
        ma = {**get_metrics(pa, gt), "scene_id": row.scene_id,
              "class_name": row.class_name, "mode": "alpha"}
        rows_a.append(ma)

        # Mode B: RVI_dp/2 — mismatch, what current codebase would produce
        pb, probb = infer_scene(model, build_rvi(row.image_path))
        mb = {**get_metrics(pb, gt), "scene_id": row.scene_id,
              "class_name": row.class_name, "mode": "rvi_dp"}
        rows_b.append(mb)

        if len(viz) < 6 and gt.any():
            viz.append({"scene_id": row.scene_id, "class_name": row.class_name,
                        "vv": build_alpha(row.image_path)[0], "gt": gt,
                        "pa": pa, "pb": pb, "proba": proba, "probb": probb,
                        "ma": ma["miou"], "mb": mb["miou"]})

    except Exception as e:
        print(f"  Skipped {row.scene_id}: {e}")
        continue

df_a = pd.DataFrame(rows_a)
df_b = pd.DataFrame(rows_b)
print(f"Done — {len(df_a)} scenes evaluated.")

In [ ]:
# Cell 6 — Results Summary
print("=" * 62)
print(" OVERALL TEST RESULTS")
print("=" * 62)
for col in ["miou", "f1", "precision", "recall"]:
    a = df_a[col].mean()
    b = df_b[col].mean()
    d = a - b
    tag = "^ alpha better" if d > 0.001 else ("v rvi better" if d < -0.001 else "~ similar")
    print(f"  {col:<12}  alpha={a:.4f}  rvi={b:.4f}  delta={d:+.4f}  {tag}")

print("\nPER-CLASS mIoU:")
for cls in df_a["class_name"].unique():
    ac = df_a[df_a["class_name"]==cls]["miou"].mean()
    bc = df_b[df_b["class_name"]==cls]["miou"].mean()
    n  = (df_a["class_name"]==cls).sum()
    print(f"  {cls:<12}  alpha={ac:.4f}  rvi={bc:.4f}  delta={ac-bc:+.4f}  n={n}")

# Save
pd.concat([df_a, df_b], ignore_index=True).to_csv(OUTPUT_DIR / "eval_results.csv", index=False)
pv = df_a.set_index("scene_id")[["class_name","miou","f1"]].rename(columns={"miou":"miou_a","f1":"f1_a"}).join(
     df_b.set_index("scene_id")[["miou","f1"]].rename(columns={"miou":"miou_b","f1":"f1_b"}))
pv["delta"] = pv["miou_a"] - pv["miou_b"]
pv.to_csv(OUTPUT_DIR / "eval_pivot.csv")
print(f"\nCSVs saved to {OUTPUT_DIR}")
pv.sort_values("delta", ascending=False).head(10)

In [ ]:
# Cell 7 — Visualisations
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def overlay(vv, pred, gt):
    rgb = np.stack([vv]*3, axis=-1).copy()
    rgb[np.logical_and(pred==1, gt==1)] = [0.1, 0.9, 0.1]  # TP green
    rgb[np.logical_and(pred==1, gt==0)] = [0.9, 0.1, 0.1]  # FP red
    rgb[np.logical_and(pred==0, gt==1)] = [1.0, 0.6, 0.0]  # FN orange
    return np.clip(rgb, 0, 1)


for i, s in enumerate(viz):
    fig, ax = plt.subplots(1, 4, figsize=(18, 4.2))
    title   = (f"{s['scene_id']} [{s['class_name']}]  "
               f"Alpha={s['ma']:.4f}  RVI_dp={s['mb']:.4f}  delta={s['ma']-s['mb']:+.4f}")
    fig.suptitle(title, fontsize=10, fontweight="bold")

    ax[0].imshow(s["vv"], cmap="gray");              ax[0].set_title("SAR VV");          ax[0].axis("off")
    ax[1].imshow(s["gt"], cmap="gray", vmin=0, vmax=1); ax[1].set_title("Ground Truth"); ax[1].axis("off")
    ax[2].imshow(overlay(s["vv"], s["pa"], s["gt"])); ax[2].set_title(f"Alpha mIoU={s['ma']:.4f}"); ax[2].axis("off")
    ax[3].imshow(overlay(s["vv"], s["pb"], s["gt"])); ax[3].set_title(f"RVI_dp mIoU={s['mb']:.4f}"); ax[3].axis("off")

    pch = [mpatches.Patch(color=c, label=l) for c,l in [([.1,.9,.1],"TP"),([.9,.1,.1],"FP"),([1,.6,0],"FN")]]
    fig.legend(handles=pch, loc="lower center", ncol=3, fontsize=9)
    fig.tight_layout()

    out = OUTPUT_DIR / f"viz_{i:02d}_{s['scene_id']}.png"
    fig.savefig(out, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  {out.name}")

print(f"\nAll outputs -> {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1024:.0f} KB)")